In [ ]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz

def normalize_text(text):
    """Normaliza texto para comparação robusta."""
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    text = "".join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))
    text = " ".join(text.split())
    return text

def extrai_valor(texto):
    """
    Busca padrão de valor monetário (R$) no texto ex: R$ 1.508.695,65 ou 1.508.695,65
    Retorna string com formato encontrado (sem R$)
    """
    if not isinstance(texto, str): return ""
    # Primeiro tenta achar com R$
    match = re.search(r'r\$ ?([\d\.]+,\d{2})', texto.lower())
    if match:
        return match.group(1)
    # Depois, só número com vírgula
    match = re.search(r'([\d\.]+,\d{2})', texto)
    return match.group(1) if match else ""
def enrich_valor_contrato(df):
    """
    Preenche campo valor_contrato, apenas se estiver vazio/NaN, usando regex no campo texto.
    """
    def pick_valor(row):
        if pd.isnull(row['valor_contrato']) or str(row['valor_contrato']).lower() in ["", "nan", "none"]:
            return extrai_valor(str(row['texto']))
        return row['valor_contrato']
    df['valor_contrato_enriched'] = df.apply(pick_valor, axis=1)
    df['valor_contrato_norm'] = df['valor_contrato_enriched'].map(normalize_text)
    return df
def match_anchors_fuzzy(df_contratos, df_csv, text_threshold=85):
    anchors = [
        'numero_contrato',
        'valor_contrato_norm',
        'objeto_contrato',
        'orgao_contratante',
        'entidade_contratada',
        'cnpj_entidade_contratada',
        'cnpj_orgao_contratante',
    ]
    # Normaliza as âncoras
    for anchor in anchors:
        if anchor not in df_contratos.columns:
            df_contratos[anchor] = ""
        df_contratos[f"{anchor}_norm"] = df_contratos[anchor].astype(str).map(normalize_text)
    df_contratos['texto_norm'] = df_contratos['texto'].map(normalize_text)
    # CSV filtrado
    df_csv_filtered = df_csv[df_csv['tipo_rel'].map(normalize_text) == 'rel_extrato_contrato'].copy()
    df_csv_filtered['texto_norm'] = df_csv_filtered['texto'].map(normalize_text)
    df_csv_filtered['tipo_ent_norm'] = df_csv_filtered['tipo_ent'].map(normalize_text)
    # Prepara mapagem por id_ato
    csv_map = {}
    for i, row in df_csv_filtered.iterrows():
        id_ato = row['id_ato']
        ent = row['tipo_ent_norm']
        val = row['texto_norm']
        if id_ato not in csv_map:
            csv_map[id_ato] = {}
        csv_map[id_ato][ent] = val
    matched_flags = []
    matched_scores = []
    matched_id_ato = []
    validation_reasons = []
    for idx, contrato in df_contratos.iterrows():
        contrato_text = contrato['texto_norm']
        best_score = 0
        best_id_ato = None
        anchor_hit = False
        anchor_used = None
        for id_ato, entgroup in csv_map.items():
            extrato_csv_text = entgroup.get('extrato_contrato', '')
            score = fuzz.token_sort_ratio(contrato_text, extrato_csv_text)
            if score >= text_threshold:
                for anchor in anchors:
                    anchor_val = contrato.get(f"{anchor}_norm", "")
                    csv_val = entgroup.get(anchor.replace('_norm',''), "")
                    if anchor_val and csv_val and anchor_val in csv_val:
                        anchor_hit = True
                        anchor_used = anchor
                        break
                if anchor_hit and score > best_score:
                    best_score = score
                    best_id_ato = id_ato
        matched_flags.append(anchor_hit)
        matched_scores.append(best_score)
        matched_id_ato.append(best_id_ato)
        validation_reasons.append(
            f'match:{anchor_used}' if anchor_hit else (f'rejected:score={best_score}')
        )
    df_contratos['multi_anchor_matched'] = matched_flags
    df_contratos['multi_anchor_score'] = matched_scores
    df_contratos['multi_anchor_id_ato'] = matched_id_ato
    df_contratos['multi_anchor_reason'] = validation_reasons
    print(f"Total contratos: {len(df_contratos)}")
    print(f"Matches validados: {matched_flags.count(True)}")
    print(f"Não-matches: {matched_flags.count(False)}")
    return df_contratos
def curadoria_base(df):
    # Campos obrigatórios mínimos
    obrigatorios = [
        'multi_anchor_id_ato',
        'valor_contrato_enriched',
        'orgao_contratante'
    ]
    df['dados_incompletos'] = df[obrigatorios].isnull().any(axis=1) | (df[obrigatorios] == '').any(axis=1)
    df_clean = df[df['multi_anchor_matched'] & ~df['dados_incompletos']]
    df_final = df_clean.drop_duplicates(subset=['multi_anchor_id_ato', 'numero_contrato'], keep='first')
    print("Duplicados removidos:", len(df_clean) - len(df_final))
    print("Registros finais (matches seguros e completos):", len(df_final))
    return df_final



In [ ]:
import pickle

def diagnose_pickle(file_path, output_txt=None, verbose=True, max_length=1000):
    """
    Diagnostica e inspeciona o conteúdo de um arquivo pickle.
    Imprime ou salva o resumo das informações, tipos de objetos, tamanhos e exemplos de dados.

    Args:
        file_path (str): Caminho do arquivo pickle.
        output_txt (str): Caminho do arquivo texto para salvar o diagnóstico (opcional).
        verbose (bool): Se True, imprime no terminal.
        max_length (int): Máximo de caracteres para exibir de exemplos.

    Returns:
        summary (str): Texto com diagnóstico resumido.
    """
    import pprint
    
    try:
        with open(file_path, "rb") as f:
            obj = pickle.load(f)
    except Exception as e:
        err_msg = f"Erro ao abrir pickle: {e}"
        print(err_msg)
        if output_txt:
            with open(output_txt, "w", encoding="utf-8") as out:
                out.write(err_msg)
        return err_msg

    summary = []
    pp = pprint.PrettyPrinter(indent=2, width=120)

    def summarize(obj, prefix="obj"):
        if isinstance(obj, dict):
            summary.append(f"{prefix} é um dict, {len(obj)} chaves.")
            keys = list(obj.keys())
            summary.append("Principais chaves: " + ", ".join(map(str, keys[:10])))
            for k in keys[:3]: # Mostra até 3 exemplos
                summary.append(f"  {prefix}[{repr(k)}] = {type(obj[k])}")
        elif isinstance(obj, list):
            summary.append(f"{prefix} é uma lista, len={len(obj)}.")
            if len(obj) > 0:
                summary.append(f"  Tipo dos elementos: {type(obj[0])}")
                example = str(obj[:3])
                summary.append(f"  Exemplos: {pp.pformat(obj[:3])[:max_length]}")
        elif isinstance(obj, tuple):
            summary.append(f"{prefix} é uma tupla, len={len(obj)}.")
            summary.append("  Conteúdo: " + pp.pformat(obj)[:max_length])
        elif isinstance(obj, set):
            summary.append(f"{prefix} é um set, len={len(obj)}.")
            summary.append("  Exemplos: " + pp.pformat(list(obj)[:5]))
        else:
            summary.append(f"{prefix} é do tipo {type(obj)}.")
            summary.append(f"  Exemplo: {str(obj)[:max_length]}")

    # Diagnóstico de objeto raiz
    summarize(obj, "obj")

    # Se obj é dict, inspeção extra nas chaves
    if isinstance(obj, dict):
        for k in list(obj.keys())[:2]:  # até 2 chaves
            summarize(obj[k], f"obj[{repr(k)}]")

    # Se obj é list ou tuple, inspeção elementos
    if isinstance(obj, (list, tuple)) and len(obj) > 0:
        for idx in range(min(2, len(obj))):
            summarize(obj[idx], f"obj[{idx}]")
    
    result = "\n".join(summary)
    if verbose:
        print(result)
    if output_txt:
        with open(output_txt, "w", encoding="utf-8") as out:
            out.write(result)
    return result

# EXEMPLO DE USO:
diagnose_pickle("corpus_by_atos_contratos.pkl", output_txt="diagnostico.txt")
with open("corpus_by_atos_contratos.pkl", "rb") as f:
    obj = pickle.load(f)

import pandas as pd

def extrai_para_df(lista):
    registros = []
    for item in lista:
        registro = {"texto": item[0]}
        for par in item[1:]:
            if isinstance(par, tuple) and len(par) == 2:
                registro[par[0]] = par[1]
        registros.append(registro)
    df = pd.DataFrame(registros)
    return df

# Exemplo de uso para contratos:
df_contratos = extrai_para_df(obj["EXTRATO_CONTRATO"])
df_contratos.head()
df_csv = pd.read_csv("DODFCorpus_contratos_fixed.csv")

# Imputa manualmente o valor nos dois casos pelo processo_gdf
import numpy as np

# Defina o mapeamento: processo_gdf -> valor que você quer inserir
impute_dict = {
    "00139-00000714/2021-59": "7.190,00",
    "00139-00000589/2021-87": "1.508.695,65"
}

# Para cada processo do dicionário acima, insere o valor nos registros correspondentes
for proc_key, valor in impute_dict.items():
    mask = df_contratos['processo_gdf'] == proc_key
    df_contratos.loc[mask, 'valor_contrato'] = valor

# Diagnóstico: confira se os valores foram preenchidos corretamente
for proc_key in impute_dict.keys():
    linha = df_contratos[df_contratos['processo_gdf'] == proc_key]
    print(f"\nDiagnóstico do processo {proc_key}:")
    print(linha[['processo_gdf', 'valor_contrato', 'texto']])

# Atualize os campos normalizados se usar pipeline posterior
df_contratos['valor_contrato_norm'] = df_contratos['valor_contrato'].astype(str).map(lambda x: x.lower().strip() if not pd.isnull(x) else "")

print("\n✅ Valores imputados com sucesso!")
df_contratos = enrich_valor_contrato(df_contratos)   # só preenche valor onde faltar
df_contratos = match_anchors_fuzzy(df_contratos, df_csv, text_threshold=85)
df_final = curadoria_base(df_contratos)

In [ ]:
# Depois de carregar df_contratos e df_csv:
df_contratos = enrich_valor_contrato(df_contratos)   # só preenche valor onde faltar
df_contratos = match_anchors_fuzzy(df_contratos, df_csv, text_threshold=85)
df_final = curadoria_base(df_contratos)
# df_final está seguro para uso!
# df_final.to_csv('base_curada_para_rag.csv', index=False)


In [ ]:
df_final.to_csv('base_curada_para_rag.csv', index=False)

In [1]:
import pandas as pd
import unicodedata
import re
import pickle
from rapidfuzz import fuzz

# ============================================================================
# FUNÇÕES AUXILIARES
# ============================================================================

def normalize_text(text):
    """Normaliza texto para comparação robusta."""
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    text = "".join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))
    text = " ".join(text.split())
    return text

def extrai_valor(texto):
    """
    Busca padrão de valor monetário (R$) no texto ex: R$ 1.508.695,65 ou 1.508.695,65
    Retorna string com formato encontrado (sem R$)
    """
    if not isinstance(texto, str): return ""
    match = re.search(r'r\$ ?([\d\.]+,\d{2})', texto.lower())
    if match:
        return match.group(1)
    match = re.search(r'([\d\.]+,\d{2})', texto)
    return match.group(1) if match else ""

def extrai_para_df(lista):
    """Extrai lista de tuplas do pickle para DataFrame."""
    registros = []
    for item in lista:
        registro = {"texto": item[0]}
        for par in item[1:]:
            if isinstance(par, tuple) and len(par) == 2:
                registro[par[0]] = par[1]
        registros.append(registro)
    df = pd.DataFrame(registros)
    return df

def enrich_valor_contrato(df):
    """
    Preenche campo valor_contrato, apenas se estiver vazio/NaN, usando regex no campo texto.
    """
    def pick_valor(row):
        if pd.isnull(row['valor_contrato']) or str(row['valor_contrato']).lower() in ["", "nan", "none"]:
            return extrai_valor(str(row['texto']))
        return row['valor_contrato']
    df['valor_contrato_enriched'] = df.apply(pick_valor, axis=1)
    df['valor_contrato_norm'] = df['valor_contrato_enriched'].map(normalize_text)
    return df

def match_anchors_fuzzy(df_contratos, df_csv, text_threshold=85):
    """Matching fuzzy com múltiplas âncoras + captura de id_dodf."""
    anchors = [
        'numero_contrato',
        'valor_contrato_norm',
        'objeto_contrato',
        'orgao_contratante',
        'entidade_contratada',
        'cnpj_entidade_contratada',
        'cnpj_orgao_contratante',
    ]
    # Normaliza as âncoras
    for anchor in anchors:
        if anchor not in df_contratos.columns:
            df_contratos[anchor] = ""
        df_contratos[f"{anchor}_norm"] = df_contratos[anchor].astype(str).map(normalize_text)
    df_contratos['texto_norm'] = df_contratos['texto'].map(normalize_text)
    
    # CSV filtrado
    df_csv_filtered = df_csv[df_csv['tipo_rel'].map(normalize_text) == 'rel_extrato_contrato'].copy()
    df_csv_filtered['texto_norm'] = df_csv_filtered['texto'].map(normalize_text)
    df_csv_filtered['tipo_ent_norm'] = df_csv_filtered['tipo_ent'].map(normalize_text)
    
    # Prepara mapagem por id_ato (inclui id_dodf)
    csv_map = {}
    id_ato_to_id_dodf = {}
    for i, row in df_csv_filtered.iterrows():
        id_ato = row['id_ato']
        id_dodf = row['id_dodf']
        ent = row['tipo_ent_norm']
        val = row['texto_norm']
        if id_ato not in csv_map:
            csv_map[id_ato] = {}
        csv_map[id_ato][ent] = val
        id_ato_to_id_dodf[id_ato] = id_dodf
    
    matched_flags = []
    matched_scores = []
    matched_id_ato = []
    matched_id_dodf = []
    validation_reasons = []
    
    for idx, contrato in df_contratos.iterrows():
        contrato_text = contrato['texto_norm']
        best_score = 0
        best_id_ato = None
        best_id_dodf = None
        anchor_hit = False
        anchor_used = None
        
        for id_ato, entgroup in csv_map.items():
            extrato_csv_text = entgroup.get('extrato_contrato', '')
            score = fuzz.token_sort_ratio(contrato_text, extrato_csv_text)
            if score >= text_threshold:
                for anchor in anchors:
                    anchor_val = contrato.get(f"{anchor}_norm", "")
                    csv_val = entgroup.get(anchor.replace('_norm',''), "")
                    if anchor_val and csv_val and anchor_val in csv_val:
                        anchor_hit = True
                        anchor_used = anchor
                        break
                if anchor_hit and score > best_score:
                    best_score = score
                    best_id_ato = id_ato
                    best_id_dodf = id_ato_to_id_dodf.get(id_ato)
        
        matched_flags.append(anchor_hit)
        matched_scores.append(best_score)
        matched_id_ato.append(best_id_ato)
        matched_id_dodf.append(best_id_dodf)
        validation_reasons.append(
            f'match:{anchor_used}' if anchor_hit else (f'rejected:score={best_score}')
        )
    
    df_contratos['multi_anchor_matched'] = matched_flags
    df_contratos['multi_anchor_score'] = matched_scores
    df_contratos['multi_anchor_id_ato'] = matched_id_ato
    df_contratos['multi_anchor_id_dodf'] = matched_id_dodf
    df_contratos['multi_anchor_reason'] = validation_reasons
    
    print(f"Total contratos: {len(df_contratos)}")
    print(f"Matches validados: {matched_flags.count(True)}")
    print(f"Não-matches: {matched_flags.count(False)}")
    return df_contratos

def curadoria_base(df):
    """Curadoria: remove duplicados e dados incompletos, retorna df_clean e df_final."""
    obrigatorios = [
        'multi_anchor_id_ato',
        'valor_contrato_enriched',
        'orgao_contratante'
    ]
    df['dados_incompletos'] = df[obrigatorios].isnull().any(axis=1) | (df[obrigatorios] == '').any(axis=1)
    df_clean = df[df['multi_anchor_matched'] & ~df['dados_incompletos']].copy()
    df_final = df_clean.drop_duplicates(subset=['multi_anchor_id_ato', 'numero_contrato'], keep='first')
    print("Duplicados removidos:", len(df_clean) - len(df_final))
    print("Registros finais (matches seguros e completos):", len(df_final))
    return df_clean, df_final

# ============================================================================
# PIPELINE COMPLETO
# ============================================================================

# 1. Carregar pickle
with open("corpus_by_atos_contratos.pkl", "rb") as f:
    obj = pickle.load(f)
df_contratos = extrai_para_df(obj["EXTRATO_CONTRATO"])

# 2. Carregar CSV
df_csv = pd.read_csv("DODFCorpus_contratos_fixed.csv")

# 3. Imputar valores manualmente para os 2 casos específicos
impute_dict = {
    "00139-00000714/2021-59": "7.190,00",
    "00139-00000589/2021-87": "1.508.695,65"
}
for proc_key, valor in impute_dict.items():
    mask = df_contratos['processo_gdf'] == proc_key
    df_contratos.loc[mask, 'valor_contrato'] = valor
print("✅ Valores imputados com sucesso!")

# 4. Enriquecer valores faltantes
df_contratos = enrich_valor_contrato(df_contratos)

# 5. Matching fuzzy
df_contratos = match_anchors_fuzzy(df_contratos, df_csv, text_threshold=85)

# 6. Curadoria (retorna df_clean e df_final)
df_clean, df_final = curadoria_base(df_contratos)

# ============================================================================
# ANÁLISE DE DUPLICADOS
# ============================================================================
print("\n" + "="*70)
print("ANÁLISE DE DUPLICADOS")
print("="*70)
duplicados = df_clean[df_clean.duplicated(subset=['multi_anchor_id_ato', 'numero_contrato'], keep=False)]
print(f"\nTotal de registros duplicados encontrados (antes da remoção): {len(duplicados)}")
if len(duplicados) > 0:
    print("\nAmostra dos duplicados:")
    print(duplicados[['multi_anchor_id_ato', 'numero_contrato', 'processo_gdf', 'valor_contrato_enriched', 'texto']].sort_values(['multi_anchor_id_ato', 'numero_contrato']).head(10))
    duplicados.to_csv("duplicados_removidos_auditoria.csv", index=False)
    print("\n✅ Duplicados salvos em 'duplicados_removidos_auditoria.csv'")

# ============================================================================
# VERSÃO FINAL - COLUNAS SELECIONADAS
# ============================================================================
print("\n" + "="*70)
print("CRIANDO VERSÃO FINAL")
print("="*70)
df_versao_final = df_final[['texto', 'objeto_contrato', 'valor_contrato_enriched', 'multi_anchor_id_ato', 'multi_anchor_id_dodf']].copy()
df_versao_final.rename(columns={'valor_contrato_enriched': 'valor_contrato'}, inplace=True)
print(f"\nDataFrame final com {len(df_versao_final)} registros")
print(df_versao_final.head())

# ============================================================================
# ANÁLISE DE id_dodf ÚNICOS
# ============================================================================
print("\n" + "="*70)
print("ANÁLISE DE id_dodf")
print("="*70)
id_dodf_unicos = df_versao_final['multi_anchor_id_dodf'].nunique()
print(f"\nTotal de id_dodf únicos no extrato final: {id_dodf_unicos}")
print(f"\nDistribuição de contratos por id_dodf:")
print(df_versao_final['multi_anchor_id_dodf'].value_counts().head(10))

# ============================================================================
# SALVAR RESULTADOS
# ============================================================================
df_versao_final.to_csv('base_curada_para_rag.csv', index=False)
print("\n✅ Base final salva em 'base_curada_para_rag.csv'")
print("\n🎉 Pipeline completo executado com sucesso!")


✅ Valores imputados com sucesso!
Total contratos: 1734
Matches validados: 1734
Não-matches: 0
Duplicados removidos: 47
Registros finais (matches seguros e completos): 1589

ANÁLISE DE DUPLICADOS

Total de registros duplicados encontrados (antes da remoção): 62

Amostra dos duplicados:
    multi_anchor_id_ato numero_contrato      processo_gdf  \
261     5_21.5.2019-R11        715/2019    310002314/2018   
278     5_21.5.2019-R11        715/2019    310002314/2018   
262     5_21.5.2019-R12        716/2019  0310-002314/2018   
279     5_21.5.2019-R12        716/2019  0310-002314/2018   
271      5_21.5.2019-R4        699/2019   310.000337/2018   
272      5_21.5.2019-R4        699/2019   310.000337/2018   
273      5_21.5.2019-R5        702/2019   310.000183/2018   
282      5_21.5.2019-R5        702/2019   310.000183/2018   
274      5_21.5.2019-R6        704/2019  0310-000298/2018   
287      5_21.5.2019-R6        704/2019  0310-000298/2018   

    valor_contrato_enriched               

In [6]:
import pandas as pd
import unicodedata
import re
import pickle
from rapidfuzz import process, fuzz
import json

# ============================================================================
# VARIÁVEIS DE CONFIGURAÇÃO (Altere aqui!)
# ============================================================================

# Arquivo JSONL com as 261 perguntas (Base A)
ARQUIVO_BASE_A_PERGUNTAS = "base_a_objeto.jsonl" 

# Arquivo CSV curado com os 1589 extratos (Base B)
ARQUIVO_BASE_B_CURADA = "base_curada_para_rag.csv"

# Threshold de similaridade (90 = 90% de confiança). 
# Se não encontrar todos, tente baixar para 85.
SCORE_THRESHOLD = 80

# ============================================================================
# FUNÇÕES AUXILIARES (Reutilizando suas funções)
# ============================================================================

def normalize_text(text):
    """Normaliza texto para comparação robusta."""
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    # Remove acentos
    text = "".join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))
    # Remove espaços duplicados e quebras de linha
    text = " ".join(text.split())
    return text

def get_root_id(id_versao_pergunta):
    """
    Extrai o ID raiz de uma pergunta.
    Ex: 'a_aquisição..._00_v0' -> 'a_aquisição..._00_'
    """
    if not isinstance(id_versao_pergunta, str):
        return ""
    # Remove a parte _v0, _v1, etc. do final
    return re.sub(r'_v\d+$', '_', id_versao_pergunta)

# ============================================================================
# FASE 1: CARGA E PREPARAÇÃO
# ============================================================================
print("--- FASE 1: Carregando e Preparando Bases ---")

# 1.1 Carregar Base A (Perguntas)
try:
    df_A_full = pd.read_json(ARQUIVO_BASE_A_PERGUNTAS, lines=True)
    print(f"✅ Base A (Perguntas) carregada: {len(df_A_full)} linhas.")
except Exception as e:
    print(f"Erro ao ler {ARQUIVO_BASE_A_PERGUNTAS}. Verifique o nome/caminho. Erro: {e}")
    exit()

# 1.2 Carregar Base B (Curada)
try:
    df_B_master = pd.read_csv(ARQUIVO_BASE_B_CURADA)
    print(f"✅ Base B (Curada) carregada: {len(df_B_master)} linhas.")
except Exception as e:
    print(f"Erro ao ler {ARQUIVO_BASE_B_CURADA}. Verifique o nome/caminho. Erro: {e}")
    exit()

# 1.3 Criar Base A "Única" (Os 87 extratos)
# Adiciona o 'id_root' para agrupar v0, v1, v2
df_A_full['id_root'] = df_A_full['id_versao_pergunta'].apply(get_root_id)

# Pega apenas a primeira ocorrência de cada 'id_root'
df_A_unicos = df_A_full.drop_duplicates(subset=['id_root'])
# Seleciona colunas que importam para a linkagem
df_A_unicos = df_A_unicos[['id_root', 'extrato', 'pdf']].copy()
print(f"✅ Base A (Única) criada: {len(df_A_unicos)} extratos únicos para linkar.")

# 1.4 Normalizar textos para comparação (usando extrato de A e texto de B)
df_A_unicos['extrato_norm'] = df_A_unicos['extrato'].apply(normalize_text)
df_B_master['texto_norm'] = df_B_master['texto'].apply(normalize_text)

# ============================================================================
# FASE 2: LINKAGEM (A -> B)
# ============================================================================
print("\n--- FASE 2: Executando Linkagem Fuzzy (A -> B) ---")

# Lista de alvos da Base B para busca
# Usar 'texto_norm' da Base B (os 1589 textos)
alvos_B = df_B_master['texto_norm'].tolist()

match_results = []

# Itera nos 87 extratos únicos da Base A
for index_A, row_A in df_A_unicos.iterrows():
    query_extrato = row_A['extrato_norm']
    
    # Encontra o melhor match para o 'extrato' da Base A dentro dos 'textos' da Base B
    # Usamos token_set_ratio: ótimo para quando um texto é subconjunto do outro
    match = process.extractOne(
        query_extrato, 
        alvos_B, 
        scorer=fuzz.token_set_ratio, 
        score_cutoff=SCORE_THRESHOLD
    )
    
    if match:
        # match = (texto_encontrado_em_B, score, indice_em_B)
        texto_match_B, score, index_B = match
        
        # Pega os IDs da Base B usando o índice encontrado
        id_ato_match_B = df_B_master.iloc[index_B]['multi_anchor_id_ato']
        id_dodf_match_B = df_B_master.iloc[index_B]['multi_anchor_id_dodf']
        
        match_results.append({
            "id_root_A": row_A['id_root'],
            "pdf_A": row_A['pdf'],
            "extrato_A_norm": query_extrato,
            "id_ato_B": id_ato_match_B,
            "id_dodf_B": id_dodf_match_B,
            "texto_match_B_norm": texto_match_B,
            "match_score": score
        })

# Cria a Base Intermediária de Linkagem
df_linkagem = pd.DataFrame(match_results)

print(f"✅ Linkagem concluída.")
print(f"Total de extratos únicos da Base A: {len(df_A_unicos)}")
print(f"Total de matches encontrados (score >= {SCORE_THRESHOLD}): {len(df_linkagem)}")

if len(df_linkagem) < len(df_A_unicos):
    print("⚠️ ATENÇÃO: Alguns extratos da Base A não encontraram match. Tente diminuir o SCORE_THRESHOLD.")
else:
    print("🎉 SUCESSO! Todos os extratos da Base A foram linkados.")

# =Não-matches
if len(df_linkagem) > 0:
    print("\n--- Validação: 5 piores matches encontrados ---")
    print(df_linkagem.sort_values(by='match_score').head().to_markdown(index=False))
else:
    print("\n⚠️ NENHUM MATCH ENCONTRADO. Verifique as colunas ou diminua o SCORE_THRESHOLD.")
    exit()

# ============================================================================
# FASE 2.5: INVESTIGAÇÃO DE FALHAS (Versão 2.1 - Corrigida)
# ============================================================================
print("\n--- FASE 2.5: Investigando os matches perdidos (v2.1) ---")

ids_encontrados = set(df_linkagem['id_root_A'])
ids_totais = set(df_A_unicos['id_root'])
ids_faltantes = ids_totais - ids_encontrados

print(f"Total de IDs faltantes: {len(ids_faltantes)}")

if len(ids_faltantes) > 0:
    print("Iniciando inspeção manual dos 3 extratos problemáticos...")
    
    # Filtra o df_A_unicos para conter apenas os que falhamos
    df_A_faltantes = df_A_unicos[df_A_unicos['id_root'].isin(ids_faltantes)]
    
    # Prepara os alvos da Base B para a busca
    alvos_B = df_B_master['texto_norm'].tolist()
    
    for index_A, row_A in df_A_faltantes.iterrows():
        query_extrato_norm = row_A['extrato_norm']
        
        print("\n" + "="*70)
        # --- CORREÇÃO AQUI ---
        print(f"INVESTIGANDO FALHA: {row_A['id_root']}") 
        # --- CORREÇÃO AQUI ---
        print(f"PDF DE ORIGEM: {row_A['pdf']}")         
        
        # Tenta encontrar o melhor match, NÃO IMPORTA O SCORE
        # Vamos ver qual é o score real, mesmo que seja 20.
        match = process.extractOne(
            query_extrato_norm, 
            alvos_B, 
            scorer=fuzz.token_set_ratio, 
            score_cutoff=0  # <-- Mudei para 0! Vamos ver TUDO.
        )
        
        if match:
            texto_match_B, score, index_B = match
            id_dodf_candidato = df_B_master.iloc[index_B]['multi_anchor_id_dodf']
            texto_candidato = df_B_master.iloc[index_B]['texto']
            
            print(f"==> MELHOR CANDIDATO ENCONTRADO (Score: {score:.2f})")
            print(f"    ID DODF Candidato: {id_dodf_candidato}")
            print(f"--- TEXTO NORMALIZADO (Base A - O que foi buscado) ---")
            print(query_extrato_norm)
            print(f"--- TEXTO NORMALIZADO (Base B - O que foi achado) ---")
            print(texto_match_B)
            print(f"--- TEXTO ORIGINAL (Base B - Para sua análise) ---")
            print(texto_candidato[:500] + "...") # Imprime 500 chars do original
            
        else:
            # Isso não deve acontecer se o score_cutoff=0, a menos que uma base esteja vazia
            print("==> ERRO CRÍTICO: Não foi possível encontrar nenhum match, nem com score 0.")

        print("="*70)

else:
    print("Nenhum ID faltante. (Esta mensagem não deveria aparecer no seu caso)")

# ============================================================================
# FASE 3: CRIAÇÃO DOS ARTEFATOS DE DADOS
# ============================================================================
print("\n--- FASE 3: Criando artefatos finais ---")

# 3.1 Salvar a Base Intermediária de Linkagem
colunas_linkagem = ['id_root_A', 'pdf_A', 'id_dodf_B', 'id_ato_B', 'match_score']
df_linkagem[colunas_linkagem].to_csv("base_intermediaria_linkagem.csv", index=False)
print(f"✅ [Artefato 1/3] Salvo: 'base_intermediaria_linkagem.csv'")

# 3.2 Criar a Base Mestra Enriquecida (Base B + PDF da Base A)
# Mapa: id_dodf -> nome_do_pdf (pega o primeiro PDF encontrado para cada id_dodf)
mapa_pdf_por_dodf = df_linkagem.drop_duplicates(subset=['id_dodf_B'])[['id_dodf_B', 'pdf_A']]
mapa_pdf_por_dodf.rename(columns={'pdf_A': 'pdf_gold'}, inplace=True)

# Faz o "left join" da Base B com o mapa de PDFs
df_B_master_enriquecido = pd.merge(
    df_B_master,
    mapa_pdf_por_dodf,
    left_on='multi_anchor_id_dodf',
    right_on='id_dodf_B',
    how='left'
)
# Limpa colunas auxiliares
df_B_master_enriquecido.drop(columns=['texto_norm', 'id_dodf_B'], inplace=True, errors='ignore')

df_B_master_enriquecido.to_csv("tabela_mestre_enriquecida.csv", index=False)
print(f"✅ [Artefato 2/3] Salvo: 'tabela_mestre_enriquecida.csv'")

# 3.3 Criar o Set de Avaliação RAG (Base A + IDs da Base B)
# Mapa: id_root -> ids_B
mapa_ids_por_root = df_linkagem[['id_root_A', 'id_dodf_B', 'id_ato_B']]

# Faz o "left join" da Base A (completa, 261 linhas) com o mapa de IDs
df_A_rag_eval = pd.merge(
    df_A_full,
    mapa_ids_por_root,
    left_on='id_root',
    right_on='id_root_A',
    how='left'
)
# Limpa colunas auxiliares
df_A_rag_eval.drop(columns=['id_root', 'id_root_A'], inplace=True, errors='ignore')
df_A_rag_eval.rename(columns={'id_dodf_B': 'id_dodf_linkado', 'id_ato_B': 'id_ato_linkado'}, inplace=True)

# Salva como JSONL
df_A_rag_eval.to_json("rag_evaluation_dataset_final.jsonl", orient='records', lines=True)
print(f"✅ [Artefato 3/3] Salvo: 'rag_evaluation_dataset_final.jsonl'")

# ============================================================================
# FASE 4: RELATÓRIO DE GAPS (O Trabalho Futuro)
# ============================================================================
print("\n--- FASE 4: Relatório de Gaps (Próximos Passos) ---")

# Usa a Base Mestra Enriquecida para o relatório
todos_id_dodf = set(df_B_master_enriquecido['multi_anchor_id_dodf'].unique())
encontrados_id_dodf = set(df_B_master_enriquecido[df_B_master_enriquecido['pdf_gold'].notnull()]['multi_anchor_id_dodf'].unique())
faltantes_id_dodf = todos_id_dodf - encontrados_id_dodf

print(f"Total de 'id_dodf' únicos na Base Mestra: {len(todos_id_dodf)}")
print(f"Total de 'id_dodf' que tiveram PDF encontrado (via Base A): {len(encontrados_id_dodf)}")
print(f"Total de 'id_dodf' FALTANTES (trabalho manual futuro): {len(faltantes_id_dodf)}")

print("\n🎉 Pipeline de linkagem concluído com sucesso!")

--- FASE 1: Carregando e Preparando Bases ---
✅ Base A (Perguntas) carregada: 261 linhas.
✅ Base B (Curada) carregada: 1589 linhas.
✅ Base A (Única) criada: 87 extratos únicos para linkar.

--- FASE 2: Executando Linkagem Fuzzy (A -> B) ---
✅ Linkagem concluída.
Total de extratos únicos da Base A: 87
Total de matches encontrados (score >= 80): 84
⚠️ ATENÇÃO: Alguns extratos da Base A não encontraram match. Tente diminuir o SCORE_THRESHOLD.

--- Validação: 5 piores matches encontrados ---
| id_root_A                                                        | pdf_A                           | extrato_A_norm                                                                                                                                                                                                                                                                                                                                                                                                      

In [7]:
import pandas as pd

ARQUIVO_MESTRE = "tabela_mestre_enriquecida.csv"
ARQUIVO_LISTA_TRABALHO = "lista_de_trabalho_pdfs_faltantes.csv"

print(f"--- Iniciando Curadoria da '{ARQUIVO_MESTRE}' ---")

df_mestre = pd.read_csv(ARQUIVO_MESTRE)

# --- 1. Relatório de Status (O que você pediu) ---
total_linhas = len(df_mestre)
linhas_com_pdf = df_mestre['pdf_gold'].notnull().sum()
linhas_sem_pdf = df_mestre['pdf_gold'].isnull().sum()

print("\n--- Relatório de Status da Tabela Mestra ---")
print(f"Total de Extratos (linhas): {total_linhas}")
print(f"Extratos COM PDF: {linhas_com_pdf}")
print(f"Extratos SEM PDF: {linhas_sem_pdf}")

# --- 2. Gerar Lista de Trabalho ---
# Pega todos os id_dodf onde a coluna 'pdf_gold' está Nula/NaN
ids_faltantes = df_mestre[df_mestre['pdf_gold'].isnull()]['multi_anchor_id_dodf'].unique()

print(f"\nTotal de 'id_dodf' únicos FALTANTES: {len(ids_faltantes)}")

# Cria um DataFrame para ser seu arquivo de trabalho
df_trabalho = pd.DataFrame(ids_faltantes, columns=['id_dodf_faltante'])

# Adiciona uma coluna vazia para você preencher
df_trabalho['pdf_encontrado_manual'] = "" 

# Salva o arquivo de trabalho
df_trabalho.to_csv(ARQUIVO_LISTA_TRABALHO, index=False)

print(f"✅ 'Lista de Trabalho' salva em: '{ARQUIVO_LISTA_TRABALHO}'")
print("Próximo passo: Abra este CSV e preencha a coluna 'pdf_encontrado_manual'.")

--- Iniciando Curadoria da 'tabela_mestre_enriquecida.csv' ---

--- Relatório de Status da Tabela Mestra ---
Total de Extratos (linhas): 1589
Extratos COM PDF: 1203
Extratos SEM PDF: 386

Total de 'id_dodf' únicos FALTANTES: 23
✅ 'Lista de Trabalho' salva em: 'lista_de_trabalho_pdfs_faltantes.csv'
Próximo passo: Abra este CSV e preencha a coluna 'pdf_encontrado_manual'.
